# Texas Inpatient PUDF 2019 and 2020: data exploration of the delivery cohort

This notebook explores the delivery cohort made by Texas_PUDF_Delivery_SMM_2019_2020.ipynb. Run that notebook first. This one only reads its output file.

Goal: see what the data can and cannot support, so we can adjust the project objectives before modeling.

Sections:
1. Load and overview
2. Who is in the cohort (Table 1 by race and ethnicity)
3. The SMM outcome: how rare it is and which indicators drive it
4. SMM subtypes by race and ethnicity
5. SMM by age, payer and cesarean
6. Crude and adjusted race gaps (a first look, not the final model)
7. Change over time (2019 vs 2020)
8. Hospitals
9. Geography (county)
10. Secondary outcomes: length of stay and in-hospital death
11. Is the planned modeling feasible?

Part 2: deeper exploration
12. Data quality check
13. Admission type
14. Principal diagnosis
15. Age by race and ethnicity
16. Payer mix by race and ethnicity
17. Delivery mode by race and ethnicity
18. Which SMM indicators happen together
19. Absolute size of the gaps (standardized rates and excess cases)
20. Summary
21. Revised objectives

Data: Texas Health Care Information Collection, Texas Department of State Health Services, Texas Inpatient Public Use Data File. Do not share the data or the outputs of this notebook outside the team, and check the Data Use Agreement before reporting small counts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import warnings
import statsmodels.api as sm

dataDir = Path("/Users/sandra/Documents/DataSets")   # same folder as the preprocessing notebook
outDir = dataDir / "output"
cohort = pd.read_parquet(outDir / "delivery_cohort_2019_2020.parquet")

attBlue, attOrange, attGray = "#009FDB", "#FF8300", "#5A6D7B"
inkColor = "#333333"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True, "figure.dpi": 110, "axes.titlesize": 11, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})

mainOutcome = "smmNoTransfusion"
raceEthOrder = list(cohort["raceEth"].cat.categories)
mainGroups = ["NH White", "Hispanic", "NH Black", "NH Asian or Pacific Islander", "NH Other"]   # groups with enough cases
cohort["los"] = pd.to_numeric(cohort["los"], errors="coerce")

def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    den = 1 + z**2 / n
    mid = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return mid - half, mid + half

def rateBy(frame, col, outcome=mainOutcome, scale=1000):
    g = frame.groupby(col, observed=True)[outcome].agg(cases="sum", deliveries="size")
    lo, hi = wilson(g["cases"], g["deliveries"])
    g["rate"] = g["cases"] / g["deliveries"] * scale
    g["ciLow"], g["ciHigh"] = lo * scale, hi * scale
    g["fewCases"] = g["cases"] < 20
    return g

def hbar(ax, tab, color=attBlue, fmt="{:.1f}", xlabel="SMM without transfusion per 1,000 deliveries (95% CI)"):
    y = np.arange(len(tab))
    v, lo, hi = tab["rate"].to_numpy(), tab["ciLow"].to_numpy(), tab["ciHigh"].to_numpy()
    ax.barh(y, v, height=0.6, color=color, xerr=[v - lo, hi - v], error_kw={"ecolor": attGray, "capsize": 2, "lw": 1})
    ax.set_yticks(y, [str(i) for i in tab.index]); ax.invert_yaxis()
    for yi, val, h in zip(y, v, hi):
        ax.text(h, yi, "  " + fmt.format(val), va="center", fontsize=8, color=inkColor)
    ax.set_xlabel(xlabel); ax.grid(axis="y", visible=False)

print(f"{len(cohort):,} delivery hospitalizations, {cohort.shape[1]} columns")
print(f"{cohort[mainOutcome].sum():,} with SMM without transfusion, {cohort['smmWithTransfusion'].sum():,} with transfusion included")

## 2. Who is in the cohort
Column percentages within each race and ethnicity group. Look for groups that differ a lot in age or payer, because those are the factors we adjust for later.

In [ ]:
def colPct(col):
    t = pd.crosstab(cohort[col].astype(str), cohort["raceEth"], normalize="columns").mul(100).round(1)
    t.index = [f"{col}: {i}" for i in t.index]
    return t

table1 = pd.concat([colPct(c) for c in ["ageCat", "payerGroup", "cesarean", "year"]])
table1.loc["deliveries (n)"] = cohort["raceEth"].value_counts().reindex(table1.columns)
display(table1)

## 3. The SMM outcome
- How rare is SMM? This decides how we handle class imbalance and which metrics we use (AUPRC and recall, not accuracy).
- Which indicators drive the composite? If one or two indicators make up most cases, the composite mostly measures those.

In [ ]:
p = cohort[mainOutcome].mean()
print(f"SMM without transfusion: {p*1000:.2f} per 1,000 deliveries, about 1 in {1/p:.0f}")
print(f"SMM with transfusion:    {cohort['smmWithTransfusion'].mean()*1000:.2f} per 1,000")
print(f"Transfusion only (no other indicator): {(cohort['smm_transfusion'] & ~cohort[mainOutcome]).sum():,} deliveries")

indCols = [c for c in cohort.columns if c.startswith("smm_")]
indCounts = cohort[indCols].sum().sort_values()
indCounts.index = [c.replace("smm_", "") for c in indCounts.index]

nInd = cohort.loc[cohort[mainOutcome], "smmCount"].clip(upper=4).value_counts().sort_index()
nInd.index = ["1", "2", "3", "4+"][:len(nInd)]

fig, ax = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [2, 1]})
colors = [attOrange if i == "transfusion" else attBlue for i in indCounts.index]
ax[0].barh(indCounts.index, indCounts.values, color=colors, height=0.7)
for yi, v in enumerate(indCounts.values):
    ax[0].text(v, yi, f"  {v:,}", va="center", fontsize=7, color=inkColor)
ax[0].set_title("Deliveries with each SMM indicator (transfusion in orange)"); ax[0].set_xlabel("deliveries")
ax[0].grid(axis="y", visible=False)
ax[1].bar(nInd.index, nInd.values, color=attBlue, width=0.6)
for xi, v in enumerate(nInd.values):
    ax[1].text(xi, v, f"{v:,}", ha="center", va="bottom", fontsize=8, color=inkColor)
ax[1].set_title("Number of indicators per SMM case"); ax[1].set_xlabel("indicators (without transfusion)")
ax[1].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
print(f"Share of SMM cases with more than one indicator: {(cohort.loc[cohort[mainOutcome], 'smmCount'] > 1).mean():.2f}")

## 4. SMM subtypes by race and ethnicity
The proposal wants to split SMM into clinical categories. I grouped the 20 non-transfusion indicators into 8 categories (my grouping, similar in spirit to El Ayadi et al. 2024, not an official CDC grouping):
- Hemorrhage: DIC, hysterectomy, shock
- Hypertensive and cerebrovascular: eclampsia, puerperal cerebrovascular disorders
- Sepsis
- Cardiac: AMI, aneurysm, cardiac arrest, heart failure during surgery, pulmonary edema or acute heart failure, rhythm conversion
- Respiratory: ARDS, ventilation, tracheostomy
- Renal: acute renal failure
- Embolism: amniotic fluid embolism, air and thrombotic embolism
- Other: anesthesia complications, sickle cell crisis

A delivery can be in more than one category. The heatmap shows the rate ratio compared with NH White (orange = higher, blue = lower). Cells marked * have fewer than 20 cases and are not reliable.

In [ ]:
smmGroups = {
    "hemorrhage": ["dic", "hysterectomy", "shock"],
    "hypertensiveCerebro": ["eclampsia", "puerperalCerebrovascular"],
    "sepsis": ["sepsis"],
    "cardiac": ["ami", "aneurysm", "cardiacArrest", "heartFailureSurgery", "pulmonaryEdemaHeartFailure", "rhythmConversion"],
    "respiratory": ["ards", "ventilation", "tracheostomy"],
    "renal": ["acuteRenalFailure"],
    "embolism": ["amnioticEmbolism", "airThromboticEmbolism"],
    "other": ["anesthesiaComplications", "sickleCellCrisis"],
}
for g, inds in smmGroups.items():
    cohort["grp_" + g] = cohort[["smm_" + i for i in inds]].any(axis=1)
grpCols = ["grp_" + g for g in smmGroups] + ["smm_transfusion"]

grpCases = cohort.groupby("raceEth", observed=True)[grpCols].sum().T
grpRate = cohort.groupby("raceEth", observed=True)[grpCols].mean().T * 10_000
grpRatio = grpRate.div(grpRate["NH White"], axis=0)
grpCases.index = grpRate.index = grpRatio.index = [c.replace("grp_", "").replace("smm_", "") for c in grpCols]
print("Cases by subtype and group:")
display(grpCases)
print("Rate per 10,000 deliveries:")
display(grpRate.round(1))

showGroups = [g for g in mainGroups if g != "NH White"] + ["NH American Indian"]
ratio, cases = grpRatio[showGroups], grpCases[showGroups]
cmap = LinearSegmentedColormap.from_list("blueGrayOrange", [attBlue, "#EEEEEE", attOrange])
cmap.set_bad("#FFFFFF")
shown = np.ma.masked_where(cases.to_numpy() < 20, np.clip(ratio.to_numpy(), 1 / 3, 3))   # color scale capped at 3
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(shown, cmap=cmap, norm=TwoSlopeNorm(vmin=1 / 3, vcenter=1, vmax=3), aspect="auto")
for i in range(ratio.shape[0]):
    for j in range(ratio.shape[1]):
        txt = f"{ratio.iat[i, j]:.2f}" + ("*" if cases.iat[i, j] < 20 else "")
        ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="#999999" if cases.iat[i, j] < 20 else inkColor)
ax.set_xticks(range(len(showGroups)), [g.replace(" or Pacific Islander", "/PI") for g in showGroups], rotation=20, ha="right")
ax.set_yticks(range(len(ratio)), ratio.index); ax.grid(False)
ax.set_title("SMM subtype rate ratio compared with NH White\n(white cells, * = fewer than 20 cases; color capped at 3)")
cb = fig.colorbar(im, ax=ax, shrink=0.8, label="rate ratio", ticks=[1 / 3, 0.5, 1, 2, 3])
cb.ax.set_yticklabels(["0.33", "0.5", "1", "2", "3+"])
plt.tight_layout(); plt.show()

# sickle cell crisis is almost only seen in Black patients, so check how much of the composite gap it explains
cohort["smmNoSickle"] = (cohort["smmCount"] - cohort["smm_sickleCellCrisis"].astype(int)) > 0
for col, lab in [(mainOutcome, "SMM without transfusion"), ("smmNoSickle", "SMM without transfusion and without sickle cell crisis")]:
    r = cohort.groupby("raceEth", observed=True)[col].mean()
    print(f"{lab}: NH Black vs NH White rate ratio {r['NH Black'] / r['NH White']:.2f}")

## 5. SMM by age, payer and cesarean
These are the main confounders (age, payer) and one factor on the causal path (cesarean). Cesarean is shown for context only. It is not an adjustment variable, because it happens during the same stay and can be a response to the complication.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ageOrder = ["<20", "20-24", "25-29", "30-34", "35-39", "40+", "18-44 (masked)"]
hbar(ax[0, 0], rateBy(cohort, "ageCat").reindex(ageOrder), xlabel="per 1,000 deliveries")
ax[0, 0].set_title("By age group (masked = HIV or drug or alcohol codes)")
payerTab = rateBy(cohort, "payerGroup").sort_values("deliveries", ascending=False)
hbar(ax[0, 1], payerTab, xlabel="per 1,000 deliveries"); ax[0, 1].set_title("By payer group")
cesTab = rateBy(cohort, "cesarean").rename(index={False: "Vaginal", True: "Cesarean"})
hbar(ax[1, 0], cesTab, xlabel="per 1,000 deliveries"); ax[1, 0].set_title("By delivery mode")

# race within the two main payers
sub = cohort[cohort["payerGroup"].isin(["Medicaid", "Private"]) & cohort["raceEth"].isin(mainGroups)]
rp = sub.groupby(["raceEth", "payerGroup"], observed=True)[mainOutcome].mean().unstack() * 1000
rp = rp.reindex(mainGroups)
y = np.arange(len(rp))
ax[1, 1].barh(y - 0.2, rp["Medicaid"], height=0.38, color=attBlue, label="Medicaid")
ax[1, 1].barh(y + 0.2, rp["Private"], height=0.38, color=attOrange, label="Private")
for yi, (m, pv) in enumerate(zip(rp["Medicaid"], rp["Private"])):
    ax[1, 1].text(m, yi - 0.2, f" {m:.1f}", va="center", fontsize=7, color=inkColor)
    ax[1, 1].text(pv, yi + 0.2, f" {pv:.1f}", va="center", fontsize=7, color=inkColor)
ax[1, 1].set_yticks(y, [g.replace(" or Pacific Islander", "/PI") for g in rp.index]); ax[1, 1].invert_yaxis()
ax[1, 1].set_title("Race and ethnicity within payer"); ax[1, 1].set_xlabel("per 1,000 deliveries")
ax[1, 1].legend(frameon=False, loc="lower right"); ax[1, 1].grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

print("SMM cases by race and ethnicity x payer group (cells for interaction terms):")
display(pd.crosstab(cohort["payerGroup"], cohort["raceEth"], values=cohort[mainOutcome], aggfunc="sum").fillna(0).astype(int))

## 6. Crude and adjusted race gaps (first look)
Logistic regression on grouped data (same result as row-level, much faster). Odds ratios compared with NH White.
- Model A: race and ethnicity only (crude).
- Model B: plus age group, payer group and year.
- Model C: plus hospital delivery volume (5 tiers).

Payer groups with very few cases (other government, other, unknown) are merged into one group for these models.

This is only a preview to see whether adjustment changes the gap. It has no comorbidities yet, which is the main thing the final model needs.

In [ ]:
hospN = cohort.groupby("hospId")["year"].transform("size") / 2   # deliveries per year at the hospital
cohort["volTier"] = pd.cut(hospN, [0, 500, 1000, 2000, 4000, np.inf], labels=["<500", "500-1k", "1k-2k", "2k-4k", "4k+"])

# rare payer groups (few or no cases) are merged so the model is stable
cohort["payerModel"] = cohort["payerGroup"].replace({"Other government": "Other or unknown", "Other": "Other or unknown",
                                                    "Unknown": "Other or unknown"})

def fitGrouped(cols):
    g = cohort.groupby(cols, observed=True)[mainOutcome].agg(cases="sum", n="size").reset_index()
    refs = {"raceEth": "NH White", "ageCat": "25-29", "payerModel": "Private", "volTier": "2k-4k"}
    X = pd.concat([pd.get_dummies(pd.Categorical(g[c].astype(str), categories=[refs[c]] + sorted(set(g[c].astype(str)) - {refs[c]})),
                                  prefix=c, drop_first=True, dtype=float) if c in refs else g[[c]].astype(float)
                   for c in cols], axis=1)
    X = sm.add_constant(X)
    y = np.column_stack([g["cases"], g["n"] - g["cases"]])
    with warnings.catch_warnings():
        # harmless here: the crude model is saturated (scale warning), and a few tiny cells in the
        # Suppressed and Unknown groups have zero cases (separation warning). The main estimates are not affected.
        warnings.simplefilter("ignore")
        res = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    ci = res.conf_int()
    out = pd.DataFrame({"OR": np.exp(res.params), "ciLow": np.exp(ci[0]), "ciHigh": np.exp(ci[1])})
    out = out[out.index.str.startswith("raceEth_")]
    out.index = out.index.str.replace("raceEth_", "")
    return out

cohort["year2020"] = (cohort["year"] == 2020).astype(int)
modelA = fitGrouped(["raceEth"])
modelB = fitGrouped(["raceEth", "ageCat", "payerModel", "year2020"])
modelC = fitGrouped(["raceEth", "ageCat", "payerModel", "year2020", "volTier"])
orTable = pd.concat({"A crude": modelA, "B + age, payer, year": modelB, "C + hospital volume": modelC}, axis=1).round(2)
display(orTable)

show = [g for g in mainGroups if g != "NH White"] + ["NH American Indian"]
y = np.arange(len(show))
fig, ax = plt.subplots(figsize=(8, 4))
for m, off, col, lab in [(modelA, -0.15, attBlue, "Crude"), (modelC, 0.15, attOrange, "Adjusted (age, payer, year, hospital volume)")]:
    r = m.reindex(show)
    ax.errorbar(r["OR"], y + off, xerr=[r["OR"] - r["ciLow"], r["ciHigh"] - r["OR"]], fmt="o", ms=7, color=col,
                ecolor=col, elinewidth=2, capsize=0, label=lab)
ax.axvline(1, color=attGray, ls="--", lw=1)
ax.set_yticks(y, [g.replace(" or Pacific Islander", "/PI") for g in show]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xticks([0.5, 1, 2, 3], ["0.5", "1", "2", "3"])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("odds ratio for SMM compared with NH White (log scale, 95% CI)")
ax.set_title("Race and ethnicity gap before and after adjustment")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 7. Change over time
The proposal plans to train on earlier data and test on later data. If 2020 differs from 2019 (COVID), a 2019 to 2020 split tests the model under a shift, not under normal conditions. The second chart shows whether the race gap itself changed.

In [ ]:
cohort["period"] = cohort["year"].astype(str) + " Q" + cohort["quarter"].astype(str)
qTab = rateBy(cohort, "period")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(len(qTab))
ax[0].fill_between(x, qTab["ciLow"], qTab["ciHigh"], color=attBlue, alpha=0.15, lw=0)
ax[0].plot(x, qTab["rate"], color=attBlue, lw=2, marker="o", ms=6)
ax[0].set_xticks(x, qTab.index, rotation=45); ax[0].set_ylabel("per 1,000 deliveries")
ax[0].set_title("SMM without transfusion per quarter (band = 95% CI)")

qr = cohort[cohort["raceEth"].isin(["NH White", "NH Black", "Hispanic"])].groupby(["period", "raceEth"], observed=True)[mainOutcome].mean().unstack()
for g, col in [("NH Black", attBlue), ("Hispanic", attOrange)]:
    rr = qr[g] / qr["NH White"]
    ax[1].plot(x, rr.to_numpy(), color=col, lw=2, marker="o", ms=6, label=g)
    ax[1].text(x[-1] + 0.15, rr.iloc[-1], g, color=inkColor, fontsize=8, va="center")
ax[1].axhline(1, color=attGray, ls="--", lw=1)
ax[1].set_xticks(x, qTab.index, rotation=45); ax[1].set_ylabel("rate ratio vs NH White")
ax[1].set_title("Race gap per quarter"); ax[1].legend(frameon=False, loc="upper left")
ax[1].set_xlim(-0.3, len(x) + 0.8)
plt.tight_layout(); plt.show()

yr = rateBy(cohort, "year")
display(yr.round(2))

## 8. Hospitals
- How many hospitals, and how big are they?
- Does SMM differ by hospital volume? Large hospitals are often referral centers that receive high risk patients, so a higher rate there does not mean worse care.
- Hospital concentration: do NH Black patients deliver in different hospitals than NH White patients? If they do, part of the race gap can be a hospital gap.
- The funnel plot shows each hospital's SMM rate against its size. Points outside the gray lines are further from the state rate than chance alone would explain.

In [ ]:
hosp = cohort.groupby("hospId").agg(deliveries=(mainOutcome, "size"), cases=(mainOutcome, "sum"),
                                     blackShare=("raceEth", lambda s: (s == "NH Black").mean()))
hosp["perYear"] = hosp["deliveries"] / 2
hosp["rate"] = hosp["cases"] / hosp["deliveries"] * 1000
print(f"{len(hosp)} hospitals. Deliveries per year: median {hosp['perYear'].median():.0f}, "
      f"25th to 75th percentile {hosp['perYear'].quantile(.25):.0f} to {hosp['perYear'].quantile(.75):.0f}, max {hosp['perYear'].max():.0f}")
print(f"Hospitals with fewer than 50 deliveries in 2 years: {(hosp['deliveries'] < 50).sum()} "
      f"({hosp.loc[hosp['deliveries'] < 50, 'deliveries'].sum()} deliveries)")

volTab = rateBy(cohort, "volTier")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
hbar(ax[0], volTab, xlabel="per 1,000 deliveries"); ax[0].set_title("SMM by hospital volume (deliveries per year)")

big = hosp[hosp["deliveries"] >= 100]
pAll = cohort[mainOutcome].mean()
nGrid = np.logspace(np.log10(big["deliveries"].min()), np.log10(big["deliveries"].max()), 200)
for z in (1.96, 3.09):
    se = np.sqrt(pAll * (1 - pAll) / nGrid)
    ax[1].plot(nGrid / 2, (pAll + z * se) * 1000, color=attGray, lw=1, ls="--" if z == 3.09 else "-")
    ax[1].plot(nGrid / 2, np.clip(pAll - z * se, 0, None) * 1000, color=attGray, lw=1, ls="--" if z == 3.09 else "-")
ax[1].scatter(big["perYear"], big["rate"], s=18, color=attBlue, alpha=0.7, edgecolor="white", linewidth=0.5)
ax[1].axhline(pAll * 1000, color=attGray, lw=1)
ax[1].set_xscale("log"); ax[1].set_xlabel("deliveries per year (log scale)"); ax[1].set_ylabel("SMM per 1,000")
ax[1].set_title("Funnel plot, hospitals with 100+ deliveries (lines: 95% and 99.8%)")
plt.tight_layout(); plt.show()

outside = ((big["rate"] / 1000 - pAll).abs() > 3.09 * np.sqrt(pAll * (1 - pAll) / big["deliveries"])).sum()
print(f"Hospitals outside the 99.8% limits: {outside} of {len(big)}")

# concentration: hospital tertiles by share of NH Black deliveries
cohort["hospBlackShare"] = cohort["hospId"].map(hosp["blackShare"])
cohort["blackServing"] = pd.qcut(cohort["hospBlackShare"].rank(method="first"), 3, labels=["low", "middle", "high"])
conc = pd.crosstab(cohort["blackServing"], cohort["raceEth"], normalize="columns")[["NH White", "NH Black", "Hispanic"]].round(3)
print("Share of each group's deliveries by hospital tertile of NH Black share (tertiles of deliveries):")
display(conc)
bs = cohort[cohort["raceEth"].isin(["NH White", "NH Black"])].groupby(["blackServing", "raceEth"], observed=True)[mainOutcome].agg(["sum", "mean"])
bs["per1000"] = bs["mean"] * 1000
print("SMM per 1,000 for NH White and NH Black within each tertile:")
display(bs["per1000"].unstack().round(2))

## 9. Geography
PAT_COUNTY is the FIPS code of the patient's county of residence (User Manual), so it can be joined to county data such as the March of Dimes maternity care deserts (Texas state FIPS 48 plus the 3 digit county code). It is the patient's county, not the hospital's.

In [ ]:
cty = cohort["county"].fillna("").astype(str).str.strip()
print(f"Counties: {cty[cty != ''].nunique()}, blank county: {(cty == '').mean():.3f}")
print("Blank county by age masking (masked HIV or drug or alcohol records have no county):")
display(cohort.assign(blankCounty=cty == "").groupby("ageMasked")["blankCounty"].mean().round(3))

ctyTab = rateBy(cohort.assign(county=cty)[cty != ""], "county").sort_values("deliveries", ascending=False)
top = ctyTab.head(12).copy()
countyNames = {"201": "Harris", "113": "Dallas", "439": "Tarrant", "029": "Bexar", "453": "Travis", "085": "Collin",
               "215": "Hidalgo", "141": "El Paso", "157": "Fort Bend", "121": "Denton", "339": "Montgomery",
               "491": "Williamson", "061": "Cameron", "479": "Webb", "303": "Lubbock", "355": "Nueces", "039": "Brazoria"}
top.index = [f"{countyNames.get(i, 'county')} ({i})" for i in top.index]
print(f"The 12 largest counties hold {top['deliveries'].sum() / len(cohort):.0%} of deliveries.")
fig, ax = plt.subplots(figsize=(8, 5))
hbar(ax, top, xlabel="per 1,000 deliveries"); ax.axvline(pAll * 1000, color=attGray, ls="--", lw=1)
ax.set_title("SMM in the 12 largest counties of residence (dashed = state)")
plt.tight_layout(); plt.show()
smallCty = (ctyTab["cases"] < 10).sum()
print(f"Counties with fewer than 10 SMM cases in 2 years: {smallCty} of {len(ctyTab)}. County-level rates need pooling or smoothing.")

## 10. Secondary outcomes: length of stay and in-hospital death

In [ ]:
los = cohort["los"]
print(f"Length of stay: median {los.median():.0f} days, 99th percentile {los.quantile(.99):.0f}, max {los.max():.0f}")
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.arange(0.5, 16.5, 1)
for flag, col, lab in [(False, attBlue, "No SMM"), (True, attOrange, "SMM")]:
    ax.hist(los[cohort[mainOutcome] == flag].clip(upper=15), bins=bins, density=True, histtype="step", lw=2, color=col, label=lab)
ax.set_xlabel("length of stay in days (15 = 15 or more)"); ax.set_ylabel("share of deliveries")
ax.set_title("Length of stay with and without SMM"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

losTab = cohort.groupby("raceEth", observed=True).agg(medianLos=("los", "median"), meanLos=("los", "mean"),
                                                      longStay7plus=("los", lambda s: (s >= 7).mean()))
display(losTab.round(3))
print("Note: length of stay is shaped by SMM and by cesarean, so it works as an outcome, not as a predictor.")

d = cohort["diedInHospital"]
print(f"In-hospital deaths: {int(d.sum())} ({d.mean()*100_000:.1f} per 100,000 deliveries)")
print(f"Deaths with SMM: {int((d & cohort[mainOutcome]).sum())}, without SMM: {int((d & ~cohort[mainOutcome]).sum())}")
print("Deaths are too few to model or to split by race reliably. Report the total, and check the DUA cell size rule before any split.")

## 11. Is the planned modeling feasible?
- Events per parameter: a common rule of thumb is at least 10 to 20 outcome events per model parameter.
- Temporal split: train on 2019, test on 2020. Each group needs enough test cases to estimate recall and calibration by group. I use 100 test cases as a rough minimum for a stable group-level metric.

In [ ]:
feas = pd.crosstab(cohort["raceEth"], cohort["year"], values=cohort[mainOutcome], aggfunc="sum").fillna(0).astype(int)
feas.columns = ["trainCases2019", "testCases2020"]
feas["testDeliveries2020"] = cohort[cohort["year"] == 2020].groupby("raceEth", observed=True).size()
feas["enoughForGroupMetrics"] = feas["testCases2020"] >= 100
display(feas)

events2019 = int(cohort.loc[cohort["year"] == 2019, mainOutcome].sum())
for nParams in (20, 60, 150):
    print(f"{nParams:>4} parameters: {events2019 / nParams:,.0f} events per parameter in the 2019 training data")
print(f"Class balance: 1 SMM case for every {1/cohort[mainOutcome].mean() - 1:.0f} deliveries without SMM")

# Part 2: deeper exploration
Sections 12 to 19 go deeper into data quality, factors known at admission, and how large the race gaps are in absolute numbers.

## 12. Data quality check
- Which fields have blanks, and are the blanks random or caused by DSHS masking?
- Where do patients live (Texas, other states, other countries)?
- Any duplicate records or impossible values?

In [ ]:
qualCols = ["sex", "race", "ethnicity", "county", "state", "zip", "payer", "princDx", "los"]
quality = pd.DataFrame({
    "blank": [cohort[c].isna().sum() for c in qualCols],
    "blankAmongMaskedAge": [cohort.loc[cohort["ageMasked"], c].isna().sum() for c in qualCols],
}, index=qualCols)
quality["blankShare"] = (quality["blank"] / len(cohort)).round(4)
quality["shareOfBlanksFromMasking"] = (quality["blankAmongMaskedAge"] / quality["blank"].replace(0, np.nan)).round(2)
display(quality)

print("Sex values:", cohort["sex"].fillna("blank").value_counts().to_dict())
print(f"Duplicate record IDs: {cohort['recordId'].duplicated().sum()}")
print(f"Length of stay: min {cohort['los'].min():.0f}, stays over 30 days {(cohort['los'] > 30).sum():,}")

# residence is made in the preprocessing notebook from PAT_STATE (User Manual): TX = Texas; AR, LA, NM, OK and
# ZZ (all other states and territories) = other US state; FC and XX = foreign country; back quote or blank = unknown
resTab = rateBy(cohort, "residence").sort_values("deliveries", ascending=False)
display(resTab[["deliveries", "cases", "rate"]].round(2))

What this shows
- All blanks in sex and ZIP come from DSHS masking: records in the masked age group (HIV or drug or alcohol codes) have no sex and no ZIP. These blanks are not random and must not be imputed. "Masked" is its own group.
- County is different: most blank counties (about 10,900 of 13,086) belong to patients who live outside Texas, which is expected because PAT_COUNTY is a Texas county code. Only about 2,200 Texas residents have a blank county.
- The 14 male and 501 unknown sex records are kept. They are coding errors or gender identity entries on real delivery records, and dropping them would not change any result.
- About 1.5% of patients live outside Texas (about 7,500 in other US states and 3,100 in another country, mostly Mexico). Their SMM rate is a bit higher, likely because they are transferred in or come for complicated care. Keep them, but a sensitivity analysis with Texas residents only is cheap.
- No duplicate record IDs.

## 13. Admission type
TYPE_OF_ADMISSION is known at admission, so unlike length of stay it is a fair predictor. Codes: 1 emergency, 2 urgent, 3 elective (mostly scheduled inductions and cesareans), 4 newborn, 5 trauma, 9 unknown. Codes 4, 5 and 9 are rare and are merged into "Other or unknown".

In [ ]:
admitLabels = {"1": "Emergency", "2": "Urgent", "3": "Elective"}
cohort["admitGroup"] = cohort["admitType"].map(admitLabels).fillna("Other or unknown")
admitOrder = ["Emergency", "Urgent", "Elective", "Other or unknown"]

print("Share of each group's deliveries by admission type:")
display(pd.crosstab(cohort["raceEth"], cohort["admitGroup"], normalize="index")[admitOrder].round(3))

groupColors = {"NH White": attGray, "Hispanic": attOrange, "NH Black": attBlue}   # same colors in every chart below
three = list(groupColors)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
hbar(ax[0], rateBy(cohort, "admitGroup").reindex(admitOrder), xlabel="per 1,000 deliveries")
ax[0].set_title("SMM by admission type")

ar = cohort[cohort["raceEth"].isin(three)].groupby(["admitGroup", "raceEth"], observed=True)[mainOutcome].mean().unstack() * 1000
ar = ar.reindex(admitOrder[:3])[three]
x = np.arange(len(ar))
for i, g in enumerate(three):
    ax[1].bar(x + (i - 1) * 0.27, ar[g], width=0.25, color=groupColors[g], label=g)
ax[1].set_xticks(x, ar.index); ax[1].set_ylabel("per 1,000 deliveries")
ax[1].set_title("Race and ethnicity within admission type"); ax[1].grid(axis="x", visible=False)
ax[1].legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()
print("NH Black vs NH White rate ratio within each admission type:")
display((ar["NH Black"] / ar["NH White"]).round(2).rename("ratio").to_frame())

What this shows
- Emergency and urgent admissions have higher SMM than elective ones (about 9 vs 6.6 per 1,000).
- NH White patients have more elective admissions (62%) than Hispanic (47%) and NH Black (50%) patients. Part of the gap may come from planned vs unplanned delivery.
- But the NH Black gap is about the same inside every admission type (1.8 to 2 times). Admission type is worth adding as a predictor, but it will not explain the gap.

## 14. Principal diagnosis
The cohort file keeps only the principal diagnosis (the main reason for the stay), not all 25 diagnosis codes. Grouping it shows which clinical conditions carry the risk. Careful:
- One code per stay, so this is not a full comorbidity list. A patient with diabetes whose principal code is about labor is not counted as diabetes here.
- Some principal codes describe a complication that happened during the stay (for example postpartum hemorrhage O72, or O75 which includes obstetric shock). These are partly the outcome, so the principal diagnosis cannot be used as a predictor as is.

In [ ]:
pdxGroups = {
    "Hypertensive disorder (O10-O16)": ["O10", "O11", "O13", "O14", "O15", "O16"],
    "Placenta previa or abruption (O44, O45)": ["O44", "O45"],
    "Multiple gestation (O30)": ["O30"],
    "Infection or other maternal disease (O98, O99)": ["O98", "O99"],
    "Diabetes (O24)": ["O24"],
    "Uterine scar or pelvic organ care (O34)": ["O34"],
    "Preterm labor (O60)": ["O60"],
    "Membranes or amniotic fluid (O40-O42)": ["O40", "O41", "O42"],
    "Fetal problem or position (O32, O35, O36, O64-O69, O76, O77)": ["O32", "O35", "O36", "O64", "O65", "O66", "O68", "O69", "O76", "O77"],
    "Complication during the stay (O71, O72, O75)": ["O71", "O72", "O75"],
    "Perineal laceration (O70)": ["O70"],
    "Normal or no indication (O80, O82)": ["O80", "O82"],
}
pdx3 = cohort["princDx"].fillna("").str[:3]
codeToGroup = {c: g for g, cs in pdxGroups.items() for c in cs}
cohort["pdxGroup"] = pdx3.map(codeToGroup).fillna("Other")

pdxTab = rateBy(cohort, "pdxGroup").sort_values("rate", ascending=False)
pdxTab["shareOfDeliveries"] = pdxTab["deliveries"] / len(cohort)
pdxTab["shareOfSmmCases"] = pdxTab["cases"] / cohort[mainOutcome].sum()
display(pdxTab[["deliveries", "cases", "rate", "shareOfDeliveries", "shareOfSmmCases"]].round(3))

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [attOrange if "during the stay" in g else attBlue for g in pdxTab.index]
hbar(ax, pdxTab, color=colors, xlabel="per 1,000 deliveries")
ax.axvline(cohort[mainOutcome].mean() * 1000, color=attGray, ls="--", lw=1)
ax.set_title("SMM by principal diagnosis group (orange = partly the outcome, dashed = all deliveries)")
plt.tight_layout(); plt.show()

print("Share of deliveries with a hypertensive principal diagnosis, by group:")
hyp = cohort["pdxGroup"].eq("Hypertensive disorder (O10-O16)")
display(hyp.groupby(cohort["raceEth"], observed=True).mean().round(3).rename("hypertensiveShare").to_frame())

What this shows
- Placenta previa or abruption (72 per 1,000), hypertensive disorders (19 per 1,000) and multiple gestation (15 per 1,000) are the highest risk groups that are mostly present before or at admission.
- Hypertensive disorders are the principal diagnosis for 10% of deliveries but 26% of SMM cases. NH Black patients have them more often (14% vs 11% for NH White, and 5% for NH Asian or Pacific Islander). This is the kind of factor the comorbidity step (adjustment step b) must include.
- Because only the principal code is kept, these counts are a floor. The real number of patients with hypertension or diabetes is higher. This confirms that the preprocessing must read all diagnosis codes to build comorbidity flags.

## 15. Age by race and ethnicity
Does the race gap change with age? If the gap grows with age, an age x race interaction may be needed in the model.

In [ ]:
ageOrderNoMask = ["<20", "20-24", "25-29", "30-34", "35-39", "40+"]
ageRace = cohort[cohort["raceEth"].isin(three)].groupby(["ageCat", "raceEth"], observed=True)[mainOutcome].agg(["sum", "size"])
ageRace["rate"] = ageRace["sum"] / ageRace["size"] * 1000
lo, hi = wilson(ageRace["sum"], ageRace["size"])
ageRace["ciLow"], ageRace["ciHigh"] = lo * 1000, hi * 1000

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(len(ageOrderNoMask))
for g in three:
    r = ageRace.xs(g, level="raceEth").reindex(ageOrderNoMask)
    ax[0].fill_between(x, r["ciLow"], r["ciHigh"], color=groupColors[g], alpha=0.12, lw=0)
    ax[0].plot(x, r["rate"], color=groupColors[g], lw=2, marker="o", ms=6, label=g)
ax[0].set_xticks(x, ageOrderNoMask); ax[0].set_xlabel("age group"); ax[0].set_ylabel("per 1,000 deliveries")
ax[0].set_title("SMM by age (band = 95% CI)"); ax[0].legend(frameon=False)

rateWide = ageRace["rate"].unstack().reindex(ageOrderNoMask)
for g in ["NH Black", "Hispanic"]:
    rr = rateWide[g] / rateWide["NH White"]
    ax[1].plot(x, rr, color=groupColors[g], lw=2, marker="o", ms=6, label=g)
ax[1].axhline(1, color=attGray, ls="--", lw=1)
ax[1].set_xticks(x, ageOrderNoMask); ax[1].set_xlabel("age group"); ax[1].set_ylabel("rate ratio vs NH White")
ax[1].set_title("Race gap by age"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
display(rateWide.round(1))

What this shows
- SMM rises with age in every group, most steeply after 35.
- The NH Black gap is present at every age (1.7 to 2.2 times). It is largest at 30 to 39 (2.2 times). At 35 to 39 it is also the largest in absolute terms (19.4 vs 8.9 per 1,000).
- The Hispanic gap is small at every age (1.1 to 1.4 times) and reverses at 40+ (0.8), where the confidence bands overlap.
- A race x age interaction is plausible but small. Test it, but do not expect it to change the main finding.

## 16. Payer mix by race and ethnicity
Payer is the main measure of income we have. If groups differ a lot in payer, payer is a strong confounder.

In [ ]:
payerShow = ["Medicaid", "Private", "Self-pay or charity"]
mix = pd.crosstab(cohort["raceEth"], cohort["payerGroup"], normalize="index").reindex(mainGroups)
mix["Other or unknown"] = 1 - mix[payerShow].sum(axis=1)
mix = mix[payerShow + ["Other or unknown"]] * 100
mixColors = [attBlue, attOrange, attGray, "#C9D1D8"]

fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(mix))
y = np.arange(len(mix))
for col, c in zip(mix.columns, mixColors):
    ax.barh(y, mix[col], left=left, color=c, height=0.6, label=col, edgecolor="white", linewidth=1.5)
    for yi, (l, v) in enumerate(zip(left, mix[col])):
        if v >= 8:
            ax.text(l + v / 2, yi, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white" if c != "#C9D1D8" else inkColor)
    left += mix[col].to_numpy()
ax.set_yticks(y, [g.replace(" or Pacific Islander", "/PI") for g in mix.index]); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("% of deliveries"); ax.grid(False)
ax.set_title("Payer mix by race and ethnicity")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.tight_layout(); plt.show()

What this shows
- About 59% of Hispanic and 58% of NH Black deliveries are paid by Medicaid, vs about 30% of NH White and 20% of NH Asian or Pacific Islander deliveries.
- So payer is a strong confounder, and it is correct to adjust for it. Section 5 showed that the NH Black gap stays inside each payer, which is why adjustment only lowers the odds ratio from 1.96 to about 1.8.
- Payer is a rough proxy for income. Medicaid in pregnancy covers a wide income range, so some confounding by income is left after adjustment.

## 17. Delivery mode by race and ethnicity
Cesarean is not an adjustment variable (it can be a response to the complication), but it is useful to see whether the gap is also present among vaginal deliveries, where a cesarean cannot be the cause.

In [ ]:
cesRate = cohort.groupby("raceEth", observed=True)["cesarean"].mean().reindex(mainGroups) * 100
print("Cesarean rate (%) by group:")
display(cesRate.round(1).rename("cesareanPct").to_frame())

dm = cohort[cohort["raceEth"].isin(three)].groupby(["cesarean", "raceEth"], observed=True)[mainOutcome].mean().unstack()[three] * 1000
dm.index = dm.index.map({False: "Vaginal", True: "Cesarean"})
fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(dm))
for i, g in enumerate(three):
    ax.bar(x + (i - 1) * 0.27, dm[g], width=0.25, color=groupColors[g], label=g)
    for xi, v in zip(x, dm[g]):
        ax.text(xi + (i - 1) * 0.27, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7, color=inkColor)
ax.set_xticks(x, dm.index); ax.set_ylabel("per 1,000 deliveries"); ax.grid(axis="x", visible=False)
ax.set_title("SMM by delivery mode and race and ethnicity")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()
print("NH Black vs NH White rate ratio:", (dm["NH Black"] / dm["NH White"]).round(2).to_dict())

What this shows
- NH Black patients have a higher cesarean rate (40% vs 34%).
- The NH Black gap is the same for vaginal and cesarean deliveries (about 1.8 times). So the gap is not caused by more cesareans.

## 18. Which SMM indicators happen together
Among deliveries with SMM, for the 10 most common indicators: the share of deliveries with the row indicator that also have the column indicator. This shows whether the composite is many separate events or a few clusters (for example a hemorrhage cluster of DIC, shock and hysterectomy).

In [ ]:
indNoTr = [c for c in cohort.columns if c.startswith("smm_") and c != "smm_transfusion"]
smmCases = cohort[cohort[mainOutcome]]
topInd = smmCases[indNoTr].sum().sort_values(ascending=False).head(10).index
m = smmCases[topInd].astype(int)
both = m.T @ m
condShare = both.div(np.diag(both), axis=0)   # P(column | row)
labels = [c.replace("smm_", "") for c in topInd]

blueSeq = LinearSegmentedColormap.from_list("blueSeq", ["#FFFFFF", attBlue])
fig, ax = plt.subplots(figsize=(8, 6.5))
shownCo = condShare.to_numpy().copy()
np.fill_diagonal(shownCo, np.nan)
im = ax.imshow(shownCo, cmap=blueSeq, vmin=0, vmax=0.5)
for i in range(len(labels)):
    for j in range(len(labels)):
        if i != j:
            v = condShare.iat[i, j]
            ax.text(j, i, f"{v*100:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 0.3 else inkColor)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right"); ax.set_yticks(range(len(labels)), labels)
ax.grid(False)
ax.set_title("Of deliveries with the row indicator, % that also have the column indicator")
fig.colorbar(im, ax=ax, shrink=0.75, label="share (capped at 50%)")
plt.tight_layout(); plt.show()
print("Number of each indicator among SMM cases:")
display(smmCases[topInd].sum().rename(lambda c: c.replace("smm_", "")).to_frame("cases"))

What this shows
- Ventilation almost always comes with ARDS (85%), so the respiratory group is mostly one event coded twice.
- Shock often comes with ARDS (42%), DIC (32%) and hysterectomy (28%): the severe hemorrhage cluster.
- DIC, hysterectomy and eclampsia mostly occur alone (no other indicator above 12%). Eclampsia sometimes comes with a cerebrovascular event (9%).
- Sepsis comes with ARDS (19%) or renal failure (12%) in some cases.
- For the model, this means the composite mixes a few distinct pathways. The subtype analysis (objective 2) is the right way to look inside it.

## 19. How big are the gaps in absolute numbers?
Odds ratios say how much higher the risk is. Public health also asks how many cases that means.
- Age-standardized rate: each group's rate if it had the same age mix as the whole cohort (direct standardization).
- Excess cases: observed cases minus the cases expected if the group had NH White rates, within the same age and payer group (indirect standardization).

In [ ]:
ageWeights = cohort["ageCat"].value_counts(normalize=True)
ageRates = cohort.groupby(["raceEth", "ageCat"], observed=True)[mainOutcome].mean().unstack()
stdRate = (ageRates * ageWeights).sum(axis=1) * 1000

whiteRates = cohort[cohort["raceEth"] == "NH White"].groupby(["ageCat", "payerModel"], observed=True)[mainOutcome].mean()
rows = []
for g in [x for x in mainGroups if x != "NH White"]:
    sub = cohort[cohort["raceEth"] == g]
    key = pd.MultiIndex.from_arrays([sub["ageCat"].astype(str), sub["payerModel"].astype(str)])
    expected = whiteRates.reindex(key).fillna(sub[mainOutcome].mean()).sum()   # rare missing cells use the group's own rate
    observed = int(sub[mainOutcome].sum())
    rows.append({"group": g, "crudeRate": sub[mainOutcome].mean() * 1000, "ageStandardizedRate": stdRate[g],
                 "observedCases": observed, "expectedAtWhiteRates": round(expected),
                 "excessCases": round(observed - expected), "excessShare": (observed - expected) / observed,
                 "SIR": observed / expected})
excess = pd.DataFrame(rows).set_index("group")
print(f"NH White: crude {cohort.loc[cohort['raceEth'] == 'NH White', mainOutcome].mean()*1000:.2f}, "
      f"age-standardized {stdRate['NH White']:.2f} per 1,000")
display(excess.round(2))

fig, ax = plt.subplots(figsize=(8, 3.4))
y = np.arange(len(excess))
ax.barh(y - 0.2, excess["observedCases"], height=0.38, color=attBlue, label="Observed")
ax.barh(y + 0.2, excess["expectedAtWhiteRates"], height=0.38, color=attGray, label="Expected at NH White rates (same age and payer)")
for yi, (o, e) in enumerate(zip(excess["observedCases"], excess["expectedAtWhiteRates"])):
    ax.text(o, yi - 0.2, f" {o:,}", va="center", fontsize=7, color=inkColor)
    ax.text(e, yi + 0.2, f" {e:,}", va="center", fontsize=7, color=inkColor)
ax.set_yticks(y, [g.replace(" or Pacific Islander", "/PI") for g in excess.index]); ax.invert_yaxis()
ax.set_xlabel("SMM cases, 2019 and 2020"); ax.grid(axis="y", visible=False)
ax.set_title("Observed vs expected SMM cases")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
plt.tight_layout(); plt.show()

What this shows
- Age standardization barely changes the rates, so age mix does not explain the gaps.
- At NH White rates for the same age and payer, NH Black patients would have had about 44% fewer SMM cases over the two years (about 470 excess cases). Hispanic patients have about 270 excess cases (13%). Their gap is small, but the group is large. NH Asian or Pacific Islander: almost no excess.
- The standardized incidence ratio (SIR, observed divided by expected) is 1.77 for NH Black and 1.14 for Hispanic, close to the odds ratios of model B in section 6 (1.83 and 1.17). The two methods agree.
- Report these as rough counts only, and check the Data Use Agreement before showing exact small numbers.

In [ ]:
gap = orTable.loc["NH Black"]
summary = {
    "deliveries": len(cohort),
    "smmPer1000": round(cohort[mainOutcome].mean() * 1000, 2),
    "transfusionOnlyShareOfSmmWith": round((cohort["smm_transfusion"] & ~cohort[mainOutcome]).sum() / cohort["smmWithTransfusion"].sum(), 2),
    "maskedAgeSmmPer1000": round(cohort.loc[cohort["ageMasked"], mainOutcome].mean() * 1000, 1),
    "hospitalsOutside998": int(outside),
    "blackCrudeOR": gap[("A crude", "OR")], "blackAdjustedOR_B": gap[("B + age, payer, year", "OR")],
    "blackAdjustedOR_C": gap[("C + hospital volume", "OR")],
    "hispanicCrudeOR": orTable.loc["Hispanic", ("A crude", "OR")], "hispanicAdjustedOR_C": orTable.loc["Hispanic", ("C + hospital volume", "OR")],
    "rate2019": round(yr.loc[2019, "rate"], 2), "rate2020": round(yr.loc[2020, "rate"], 2),
    "blackRatioNoSickle": round(cohort.loc[cohort["raceEth"] == "NH Black", "smmNoSickle"].mean()
                                / cohort.loc[cohort["raceEth"] == "NH White", "smmNoSickle"].mean(), 2),
    "hospitals": len(hosp), "obUnitShare": round(cohort["obUnit"].mean(), 3),
    "deaths": int(cohort["diedInHospital"].sum()),
}
pd.Series(summary, name="value").to_frame()

## 20. Summary: what the exploration shows

Outcome
- SMM without transfusion is rare: 7.7 per 1,000 deliveries (about 1 in 130), 5,531 cases. This is a strongly imbalanced problem, so accuracy is useless as a metric. Use AUPRC, recall at a fixed alert rate, and calibration.
- 62% of the SMM-with-transfusion cases are transfusion only. The sensitivity outcome is mostly a transfusion outcome, which supports keeping SMM without transfusion as the main outcome.
- 21% of SMM cases have more than one indicator.

Race and ethnicity
- NH Black patients have about 2 times the SMM odds of NH White patients (OR 1.96). After age, payer, year and hospital volume it is still 1.72. Most of the gap is not explained by these factors.
- Hispanic: small gap (1.19 crude, 1.09 adjusted). NH Asian or Pacific Islander: no gap after adjustment. NH Other: 1.34 adjusted, but this is a mixed group and hard to interpret.
- NH American Indian (10 cases), Suppressed and Unknown are too small for their own estimates.
- The Black and White gap is the same inside Medicaid (11.9 vs 7.2 per 1,000) and inside private insurance (11.6 vs 5.8). Private insurance does not close it.

Subtypes
- The NH Black gap is largest for renal failure (3.5 times), embolism, respiratory, sepsis, cardiac and hypertensive or cerebrovascular events (about 2 times), and smallest for hemorrhage (1.4 times).
- The "other" category is 14.8 times higher for NH Black patients because it contains sickle cell crisis, a condition that mostly affects Black patients. It explains only a small part of the overall gap (the rate ratio goes from 1.95 to 1.88 without it), but it should be reported separately.
- Hispanic patients have a higher rate mainly for sepsis (1.8 times) and transfusion (1.5 times).
- Only NH White, Hispanic and NH Black have enough cases in every subtype. Subtype results for the other groups are not reliable.

Other risk factors
- Age 35 and older, Medicare (usually disability), and the masked HIV or drug or alcohol group (24.9 per 1,000, about 3 times the average) have the highest rates.
- Cesarean deliveries have about 4 times the rate of vaginal deliveries, but cesarean can be a response to the complication, so it is not an adjustment variable.

Hospitals and place
- 252 hospitals. 61% of NH Black deliveries happen in the third of hospitals with the highest NH Black share. The Black and White gap is present inside every hospital tertile, so hospital choice alone does not explain it.
- SMM is highest in the largest hospitals (10.4 per 1,000 in hospitals with 4,000+ deliveries a year). These are referral centers, so volume reflects case mix, not only quality.
- 26 of 229 hospitals are outside the 99.8% funnel limits. Hospitals differ more than chance, so models need to account for clustering by hospital.
- 198 of 254 counties have fewer than 10 SMM cases in 2 years. County-level analysis needs pooling, for example the 11 Public Health Regions.

Time
- SMM rose from 7.4 (2019) to 8.0 (2020) per 1,000. The NH Black gap moved between 1.7 and 2.2 per quarter. A 2019 train and 2020 test split tests the model under a shift.

Data limits found
- The obstetric unit flag (SPEC_UNIT = B) is on only 65% of deliveries, so it cannot define the cohort.
- The cohort file has no comorbidities yet. The proposal's adjustment for health status needs them.
- APR-DRG severity and risk of mortality are in the file but are computed from all diagnoses, including the SMM codes, so they leak the outcome and cannot be used for adjustment.
- In-hospital deaths: 46, of which 41 had SMM. Too few to model.

Part 2 (sections 12 to 19)
- Blank sex and ZIP come only from DSHS masking, and most blank counties belong to non-Texas residents. None of these blanks should be imputed.
- 1.5% of patients live outside Texas and have a slightly higher rate. A Texas-residents-only sensitivity check is cheap.
- Admission type is known at admission and is a fair new predictor (elective 6.6 vs emergency 8.9 per 1,000). NH White patients have more elective admissions, but the NH Black gap is the same inside each admission type.
- Hypertensive disorders are the principal diagnosis for 10% of deliveries but 26% of SMM cases, and are more common in NH Black patients. Placenta previa or abruption has the highest rate (72 per 1,000). Comorbidity flags from all diagnosis codes are needed.
- The NH Black gap is present at every age and largest at 30 to 39 (2.2 times). It is also the same for vaginal and cesarean deliveries (1.8 times), so more cesareans do not explain it.
- Payer mix differs a lot (Medicaid: 59% Hispanic, 58% NH Black, 30% NH White), so payer is a real confounder.
- Ventilation is almost always coded with ARDS, and shock clusters with DIC and hysterectomy. The composite mixes a few pathways.
- At NH White rates for the same age and payer, NH Black patients would have about 470 fewer SMM cases in 2 years (44% of their cases), Hispanic patients about 270 fewer.

## 21. Revised project objectives

Main objective (unchanged in spirit): measure racial and ethnic disparities in SMM among Texas delivery hospitalizations (2019 and 2020) and test whether they remain after risk adjustment.

1. Disparity estimation (primary)
- Outcome: SMM without transfusion. Sensitivity: with transfusion, without sickle cell crisis, 2019 only, and Texas residents only.
- Report excess cases (observed minus expected at NH White rates) next to the odds ratios, so the size of the gap is clear in numbers.
- Compare Hispanic, NH Black, NH Asian or Pacific Islander and NH Other with NH White. NH American Indian, Suppressed and Unknown are reported descriptively only.
- Adjust in steps and report how the odds ratio changes at each step: (a) age, payer, year; (b) plus comorbidities present at delivery (an obstetric comorbidity index); (c) plus hospital (volume tier and a hospital random effect).
- New: test whether the Black and White gap exists within hospitals (hospital fixed or random effects), since patients are concentrated in different hospitals.

2. Complication pathways
- Report SMM subtypes for NH Black and Hispanic compared with NH White, using the 7 clinical groups (hemorrhage, hypertensive or cerebrovascular, sepsis, cardiac, respiratory, renal, embolism). Sickle cell crisis is reported on its own.
- Not done for groups with fewer than 20 cases per subtype.

3. Prediction and fairness
- Predict SMM using only information known at admission: age, payer, admission type, comorbidities, hospital volume and type, and year. Length of stay, discharge status, cesarean and APR-DRG severity are not predictors (leakage).
- Compare logistic regression or LASSO with gradient boosting.
- Main split: train 2019, test 2020 (a test under the COVID shift). Add a random split within years to see how much of any drop is caused by the shift.
- Metrics: AUPRC, recall at a fixed alert rate (for example the top 5% of risk), calibration. Report them by race and ethnicity only for groups with at least 100 test cases (NH White, Hispanic, NH Black, NH Asian or Pacific Islander, NH Other).
- Race interaction terms: race x payer only for Medicaid and private, and only for NH White, Hispanic and NH Black (other cells are too small).

4. Hospitals and regions (replaces the "if time allows" goal)
- Hospital volume tiers and the share of NH Black patients per hospital, as described in section 8.
- Region: 11 Public Health Regions instead of single counties. The March of Dimes maternity care desert data stays optional, joined on county FIPS.

Secondary outcomes
- Length of stay, as an outcome only, by race and SMM status.
- In-hospital death: total count and share with SMM only. No model and no split by race (46 deaths).

Changes from the proposal
- Cohort is defined by diagnosis, DRG and procedure codes, not by the obstetric unit flag (on only 65% of deliveries) and not by revenue codes.
- No imputation needed for age or length of stay: age is an age group code with no blanks, and length of stay is not a predictor.
- No z-score scaling: all predictors are categorical or counts.
- "Hospital obstetric-unit volume" becomes the number of deliveries per hospital per year, computed from the cohort.
- Mortality becomes descriptive only.

Next preprocessing work (Member 1)
- Keep the diagnosis codes, or compute comorbidity flags while reading the files (for example chronic hypertension, pregestational diabetes, obesity, asthma, substance use, prior cesarean, multiple gestation, placenta previa). This is needed for adjustment step (b).
- Add PUBLIC_HEALTH_REGION to the cohort.
- Optional: the facility type file in each zip, for hospital type.